In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Load NetCDF datasets (datasets not included in this repository)
cape_ds = xr.open_dataset("/path/to/monthly_cape.nc")
aod_ds = xr.open_dataset("/path/to/aod_regridded.nc")
light_ds = xr.open_dataset("/path/to/LISOTD.nc", decode_times=False)          
aod = aod_ds["__xarray_dataarray_variable__"]

# Harmonize coordinate names
cape = cape_ds["cape"].rename({ 'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
light = light_ds["LRMTS_COM_FR"].rename({'Latitude': 'lat', 'Longitude': 'lon', 'Month_since_Jan_95': 'time'})
light = light.assign_coords( time=pd.date_range( start="1995-01-01", periods=light.sizes["time"], freq="MS" ))
cape = cape.transpose("time", "lat", "lon")
light = light.transpose("time", "lat", "lon")

# Interpolate all datasets onto AOD grid
cape_interp = cape.interp( time=aod.time,lat=aod.lat, lon=aod.lon)
light_interp = light.interp( time=aod.time, lat=aod.lat, lon=aod.lon)

# Compute spatial averages
light_mean = light_interp.mean(dim=["lat", "lon"])
cape_mean = cape_interp.mean(dim=["lat", "lon"])
aod_mean = aod.mean(dim=["lat", "lon"])

cape_month = cape_mean.groupby("time.month").mean()
aod_month = aod_mean.groupby("time.month").mean()
light_month = light_mean.groupby("time.month").mean()

# Calculate monthly climatology and variability
aod_std = aod_mean.groupby("time.month").std()
cape_std = cape_mean.groupby("time.month").std()
light_std = light_mean.groupby("time.month").std()

months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
x = np.arange(12)
width = 0.35

# Generate multi-axis climatology plot
fig, ax1 = plt.subplots(figsize=(10,5))

ax1.bar(x-width/2, light_month, width, yerr=light_std.values,
    capsize=3, color='red', edgecolor='black', label='Lightning')

ax1.set_ylabel("LFRD (flashes km$^{-2}$ month$^{-1}$)",color='red')

ax1.tick_params(axis='y', colors='red')
ax1.set_xticks(x)
ax1.set_xticklabels(months)
ax2 = ax1.twinx()

ax2.bar(x+width/2, cape_month, width, yerr=cape_std.values,
    capsize=3, color='blue', edgecolor='black', label='CAPE')

ax2.set_ylabel("CAPE (J kg$^{-1}$)",color='blue')

ax2.tick_params(axis='y', colors='blue')
ax2.set_ylim(bottom=0)
ax3 = ax1.twinx()

# Move third axis further right
ax3.spines["right"].set_position(("axes", 1.12))
ax3.errorbar( x, aod_month, yerr=aod_std, color='green',
              marker='o', linestyle='-', linewidth=2, capsize=3, label='AOD')

ax3.set_ylabel("AOD", color='green')
ax3.tick_params(axis='y', colors='green')
plt.title("Malawi")
h1,l1 = ax1.get_legend_handles_labels()
h2,l2 = ax2.get_legend_handles_labels()
h3,l3 = ax3.get_legend_handles_labels()

ax1.legend(h1+h2+h3, l1+l2+l3, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
Monthly climatology analysis of Lightning, CAPE and Aerosol Optical Depth (AOD).

This notebook demonstrates:
- Loading NetCDF datasets with xarray
- Harmonising spatial and temporal coordinates
- Interpolating datasets onto a common grid
- Computing monthly climatologies and spatial averages
- Producing a multi-axis climatology figure

Note:
The original datasets are not included because they form part of ongoing thesis research.